# Stage 2.5 executor scheduling paired screen

This notebook runs the first comparison requested for the executor packet:
control = underfoot baseline plus deadline guards; treatment = control plus
persistent worker queues. Schedule-informed hiring is a separate selectable
follow-up and is disabled by default. The neural manager and BC-E opponent are
unchanged: P-final stochastic JAX sampling, `E_LEGACY`, standard/copy opening.

The panel is 16 ordered environment seeds × both seats = 32 seed/seat games
per arm on the verified `combined_wheat3` variant. The wrapper keeps both seats
uses four independent child processes, and preserves the full seed list in
every manifest and filtered preflight. Use Kaggle Secrets to provide
`GITHUB_TOKEN`; never print or persist the token.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, tempfile, time

REPO_URL = 'https://github.com/BillXu21/Kaggriculture.git'
EXPERIMENT_REF = 'codex/stage25-upkeep-ablation'
CODE_SHA = '3431dbf065044a66ea469884dc85e68f74e72769'
PPO_CHECKPOINT = Path('/kaggle/working/interactive_curriculum_0acfe858/runs/P_bce_fullspace_lr3e5_from_Ou10_s43049/final.npz')
BC_E_CHECKPOINT = Path('/kaggle/input/datasets/billll/v0-bc-e/best.pt')
SEEDS = [144368101, 309507, 615013, 918079, 1221109, 1524137, 1827169, 2130193, 2433221, 2736251, 3039283, 3342311, 3645341, 3948373, 4251401, 2112243121]
MASTER_SEED = 25
VARIANTS = ['combined_wheat3']
RUN_HIRING_FOLLOWUP = False
PROCESSES = 4
E_HISTORY_VERSION = 'E_LEGACY'
BACKEND = 'official'

RUN_ROOT = Path('/kaggle/working') / ('stage25_executor_schedule_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
REPO = RUN_ROOT / 'repo'
RUN_ROOT.mkdir(parents=True, exist_ok=False)
if len(SEEDS) != 16 or len(set(SEEDS)) != 16:
    raise ValueError('SEEDS must be the complete ordered 16-seed panel')
for path in (PPO_CHECKPOINT, BC_E_CHECKPOINT):
    if not path.is_file():
        raise FileNotFoundError(f'Missing editable checkpoint path: {path}')
print('Run root:', RUN_ROOT)
print('Planned seed/seat games per variant per arm:', len(SEEDS) * 2)


In [ ]:
# Authenticate through a short-lived askpass process. The secret is never in the URL/output.
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Kaggle Secret GITHUB_TOKEN is empty')
try:
    with tempfile.TemporaryDirectory(prefix='stage25_askpass_') as tmp:
        askpass = Path(tmp) / 'askpass.py'
        askpass.write_text('import os, sys\nprint("x-access-token" if "username" in sys.argv[1].lower() else os.environ["STAGE25_TOKEN"])\n')
        env = {**os.environ, 'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'STAGE25_TOKEN': token}
        subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', EXPERIMENT_REF, REPO_URL, str(REPO)], env=env, check=True, timeout=180)
        subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CODE_SHA], cwd=REPO, env=env, check=True, timeout=180)
        subprocess.run(['git', 'checkout', '--detach', CODE_SHA], cwd=REPO, env=env, check=True, timeout=30)
finally:
    token = None
    if 'env' in globals():
        env.pop('STAGE25_TOKEN', None)
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
if actual_sha != CODE_SHA:
    raise RuntimeError(f'checkout mismatch: {actual_sha} != {CODE_SHA}')
print('Pinned source:', actual_sha)


In [ ]:
# Keep the existing Kaggle JAX/JAXLIB installation; constrain each child to one CPU.
EVAL_ENV = os.environ.copy()
EVAL_ENV.update({
    'PYTHONUNBUFFERED': '1', 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1', 'NUMEXPR_NUM_THREADS': '1',
    'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
    'XLA_FLAGS': '--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1',
    'JAX_PLATFORMS': 'cpu', 'PYTHONPATH': str(REPO),
})

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

tracked = [REPO / 'executor_v0/agent.py', REPO / 'executor_v0/foreman.py', REPO / 'executor_v0/scheduler.py', REPO / 'executor_v0/hiring.py', REPO / 'tools/evaluate_stage25_upkeep.py', REPO / 'tools/run_stage25_upkeep_sharded.py']
source_hashes = {str(path.relative_to(REPO)): sha256(path) for path in tracked}
patch_hash = hashlib.sha256(subprocess.check_output(['git', 'diff', '--binary', 'HEAD'], cwd=REPO)).hexdigest()
print(json.dumps({'source_commit': actual_sha, 'source_hashes': source_hashes, 'patch_sha256': patch_hash, 'ppo_sha256': sha256(PPO_CHECKPOINT), 'bc_e_sha256': sha256(BC_E_CHECKPOINT)}, indent=2))

# This preflight checks identity/sharding before checkpoint loading or games.
preflight = [sys.executable, '-m', 'tools.run_stage25_upkeep_sharded', '--preflight-only', '--output-dir', str(RUN_ROOT / 'preflight'), '--backend', BACKEND, '--e-history-version', E_HISTORY_VERSION, '--master-seed', str(MASTER_SEED), '--processes', str(PROCESSES), '--seeds', *map(str, SEEDS), '--variants', *VARIANTS, '--underfoot-first', '--deadline-safe-planting', '--deadline-safe-hiring']
subprocess.run(preflight, cwd=REPO, env=EVAL_ENV, check=True)
print('Preflight preserved the complete ordered seed list and 25-based episode IDs.')

# Check the exact stochastic P-final/BC-E reconstruction without changing policy code.
probe = 'import jax; from bc_manager_jax.checkpoint import load_torch_checkpoint; from rl_manager.ppo_checkpoint import load_ppo_checkpoint; print(jax.__version__)'
subprocess.run([sys.executable, '-c', probe, str(PPO_CHECKPOINT), str(BC_E_CHECKPOINT)], cwd=REPO, env=EVAL_ENV, check=True)
print('Checkpoint import preflight passed; no evaluation games have run yet.')


In [ ]:
def run_arm(name, extra_flags, resume=False):
    output = RUN_ROOT / name
    command = [sys.executable, '-m', 'tools.run_stage25_upkeep_sharded', '--checkpoint', str(PPO_CHECKPOINT), '--e-checkpoint', str(BC_E_CHECKPOINT), '--output-dir', str(output), '--backend', BACKEND, '--e-history-version', E_HISTORY_VERSION, '--master-seed', str(MASTER_SEED), '--processes', str(PROCESSES), '--seeds', *map(str, SEEDS), '--variants', *VARIANTS, *extra_flags]
    if resume:
        command.append('--resume')
    if '--capture-dir' in command:
        raise AssertionError('capture must remain off for the default paired screen')
    (RUN_ROOT / (name + '.command.json')).write_text(json.dumps({'command': command, 'source_commit': actual_sha, 'capture_neutrality': 'capture disabled; executor actions are computed before passive snapshots'}, indent=2))
    print(f'Starting {name}: {len(SEEDS) * 2 * len(VARIANTS)} evaluator rows', flush=True)
    started = time.monotonic()
    process = subprocess.Popen(command, cwd=REPO, env=EVAL_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    log_path = RUN_ROOT / (name + '.log')
    with log_path.open('w') as log:
        for line in process.stdout:
            log.write(line); log.flush(); print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'{name} failed with {code}; partial outputs remain at {output}')
    print(f'{name} completed in {(time.monotonic() - started) / 60:.1f} minutes', flush=True)
    return output

control_flags = ['--underfoot-first', '--deadline-safe-planting', '--deadline-safe-hiring']
treatment_flags = control_flags + ['--persistent-worker-queues']
control = run_arm('control_underfoot_deadline', control_flags)
treatment = run_arm('treatment_persistent_queues', treatment_flags)
if RUN_HIRING_FOLLOWUP:
    hiring = run_arm('followup_schedule_informed_hiring', treatment_flags + ['--schedule-informed-hiring'])
else:
    hiring = None
    print('Hiring follow-up is selectable but disabled.')


In [ ]:
# Validate merged identities and report the requested paired metrics.
import pandas as pd
from IPython.display import display, FileLink

def report(name, output):
    manifest = json.loads((output / 'manifest.json').read_text())
    if manifest['status'] != 'complete' or manifest['ordered_seeds'] != SEEDS:
        raise RuntimeError(f'incomplete or reordered {name} output')
    rows = [json.loads(line) for line in (output / 'games.jsonl').read_text().splitlines() if line.strip()]
    expected = len(SEEDS) * 2 * len(VARIANTS)
    if len(rows) != expected or any(row['statuses'] != ['DONE', 'DONE'] for row in rows):
        raise RuntimeError(f'{name} has {len(rows)} rows; expected {expected} completed rows')
    frame = pd.DataFrame(rows)
    summary = frame.groupby('variant', sort=False).agg(games=('bank', 'size'), candidate_bank=('bank', 'mean'), opponent_bank=('opponent_bank', 'mean'), margin=('margin', 'mean'))
    telemetry = json.loads((output / 'telemetry.json').read_text())
    print(name, 'manifest status:', manifest['status'])
    display(summary.round(1))
    for field in ('completed_work', 'missed_maintenance', 'travel_abandonment', 'hiring_cost', 'scheduler_runtime'):
        print(field, telemetry['fields'][field])
    return frame

control_rows = report('control', control)
treatment_rows = report('treatment', treatment)
all_frames = [control_rows.assign(arm='control'), treatment_rows.assign(arm='treatment')]
if hiring is not None:
    hiring_rows = report('hiring follow-up', hiring)
    all_frames.append(hiring_rows.assign(arm='schedule-informed-hiring'))
combined = pd.concat(all_frames, ignore_index=True)
combined.to_csv(RUN_ROOT / 'paired_results.csv', index=False)
bundle = shutil.make_archive(str(RUN_ROOT / 'stage25_executor_schedule_results'), 'zip', root_dir=RUN_ROOT)
display(FileLink(bundle))
print('Bootstrap unit: each bootstrap_groups.json entry contains both seats for one environment seed.')
print('Promotion authority remains official kaggle_environments==1.32.7; this notebook does not promote a result.')

# Resume validation recipe (safe after an interrupted run):
# run_arm('treatment_persistent_queues', treatment_flags, resume=True)
# The wrapper checks manifest configuration/checkpoint hashes and reuses only complete, identity-validated shards.
